# 2. Извлечение текстовых эмбеддингов

Берём предобученную модель `all-MiniLM-L6-v2` (sentence-transformers) и прогоняем через неё чанки субтитров.

Каждое видео → последовательность эмбеддингов (по одному на чанк) → вход для Transformer.

In [1]:

import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import yaml
import warnings
warnings.filterwarnings("ignore")

from src.embeddings import extract_embeddings

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_DIR = Path("../data")
EMB_MODEL = cfg["embedding"]["model_name"]
BATCH_SIZE = cfg["embedding"]["batch_size"]
MAX_SEQ_LEN = cfg["model"]["max_seq_len"]
EMB_DIM = cfg["embedding"]["embedding_dim"]

print(f"Embedding model: {EMB_MODEL}")
print(f"Configured embedding dim: {EMB_DIM}")
print(f"Max seq len: {MAX_SEQ_LEN}")


c:\Users\Пользователь\.pyenv\pyenv-win\versions\3.10.4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model: all-MiniLM-L6-v2
Configured embedding dim: 384
Max seq len: 10


In [2]:

model = SentenceTransformer(EMB_MODEL)
actual_dim = model.get_sentence_embedding_dimension()
print(f"Model loaded: {EMB_MODEL}")
print(f"Model embedding dimension: {actual_dim}")
print(f"Max sequence length: {model.max_seq_length}")

if actual_dim != cfg["model"]["d_model"]:
    raise ValueError(
        f"Model emits {actual_dim} dims, but Transformer d_model is "
        f"{cfg['model']['d_model']}. Update configs/config.yaml."
    )


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 501.06it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Model embedding dimension: 384
Max sequence length: 256


## 2.1 Функция извлечения эмбеддингов

In [3]:

print("extract_embeddings imported from src.embeddings")


extract_embeddings imported from src.embeddings


## 2.2 Извлечение эмбеддингов для train / val / test

In [4]:
df_train = pd.read_parquet(DATA_DIR / "train.parquet")
df_val = pd.read_parquet(DATA_DIR / "val.parquet")
df_test = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")

Train: 11,183, Val: 2,396, Test: 2,397


In [5]:
print("=== Train ===")
train_emb, train_labels, train_lens = extract_embeddings(df_train, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {train_emb.shape}")

print("\n=== Val ===")
val_emb, val_labels, val_lens = extract_embeddings(df_val, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {val_emb.shape}")

print("\n=== Test ===")
test_emb, test_labels, test_lens = extract_embeddings(df_test, model, BATCH_SIZE, MAX_SEQ_LEN)
print(f"Shape: {test_emb.shape}")

=== Train ===
Total chunks to encode: 108,309


Batches: 100%|██████████| 424/424 [01:26<00:00,  4.89it/s]


Shape: (11183, 10, 384)

=== Val ===
Total chunks to encode: 23,231


Batches: 100%|██████████| 91/91 [00:18<00:00,  4.88it/s]


Shape: (2396, 10, 384)

=== Test ===
Total chunks to encode: 23,159


Batches: 100%|██████████| 91/91 [00:18<00:00,  4.98it/s]


Shape: (2397, 10, 384)


## 2.3 Сохранение эмбеддингов

In [6]:
np.savez_compressed(
    DATA_DIR / "embeddings.npz",
    train_emb=train_emb,
    train_labels=train_labels,
    train_lens=train_lens,
    val_emb=val_emb,
    val_labels=val_labels,
    val_lens=val_lens,
    test_emb=test_emb,
    test_labels=test_labels,
    test_lens=test_lens,
)

file_size = (DATA_DIR / "embeddings.npz").stat().st_size / (1024**2)
print(f"Saved embeddings.npz ({file_size:.1f} MB)")
print(f"Train: {train_emb.shape}, Val: {val_emb.shape}, Test: {test_emb.shape}")
print("\nEmbedding extraction complete! Upload embeddings.npz to Kaggle Dataset.")

Saved embeddings.npz (210.1 MB)
Train: (11183, 10, 384), Val: (2396, 10, 384), Test: (2397, 10, 384)

Embedding extraction complete! Upload embeddings.npz to Kaggle Dataset.
